## B2 · Columns and the index

Selecting, renaming, reordering, dropping and retyping columns, moving columns in and out of the index, and reindexing.

In [270]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# Plot defaults: wide 16:9 figures, sharp ("retina") output and slightly smaller text
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.figsize": (6.4, 3.6),
    "figure.dpi": 100,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
})

# Same data as sns.load_dataset('penguins'), saved locally so it works offline
df_penguins = pd.read_csv("data/penguins.csv")

### Selecting columns

#### Selecting one column

**B2.01**

`df['age']  or  df.age`

Square brackets with a column name allow us to pick out one column as a Series. It is useful for calculating a statistic or plotting one variable. Alternatively, dot notation `df.age` works when the name is simple.

A single column comes out as a Series. Dot notation (`df.age`) works too, but only if the name has no spaces and does not clash with a method name, so square brackets are safer.

In [271]:
df_students = pd.DataFrame({"name": ["Ana", "Bo", "Cy"], "age": [21, 25, 22], "height": [165, 180, 172]})

In [272]:
# display the example data
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180
2,Cy,22,172


In [273]:
df_students["age"]

0    21
1    25
2    22
Name: age, dtype: int64

In [274]:
type(df_students["age"])

pandas.core.series.Series

In [275]:
df_students.age.tolist()     # dot notation: same values (this book uses the brackets form)

[21, 25, 22]

*See also:* Selecting several columns (B2) · Creating a Series (B1) · KeyError: key not found (A1) · Selecting one row by label (B3)

#### Selecting several columns

**B2.02**

`df[['age', 'height']]`

Square brackets with a *list* of names allow us to select several columns as a new DataFrame. It is useful for focusing on the variables that are needed. Alternatively, `df.loc[:, ['a', 'b']]` does the same.

Note the **double** brackets: a list of names inside the selector. The result is a DataFrame, so this can also be used to put the columns in a new order.

In [276]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo", "Cy"], "age": [21, 25, 22], "height": [165, 180, 172]})
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180
2,Cy,22,172


In [277]:
df_students[["age", "height"]]

,age,height
0,21,165
1,25,180
2,22,172


In [278]:
type(df_students['age'])        # one name  -> Series

pandas.core.series.Series

In [279]:
type(df_students[['age']])      # a list    -> DataFrame

pandas.core.frame.DataFrame

*See also:* Selecting one column (B2) · Reordering columns by listing them (B2)

### Labels

#### Renaming columns with a dictionary

**B2.03**

`df.rename(columns={'old': 'new'})`

The `rename(columns={...})` method allows us to change the names of chosen columns. It is useful for fixing a few awkward names. Alternatively, assign a whole list to `df.columns` to rename everything.

`rename` returns a **new** DataFrame, so assign it back (or chain it). Only the columns that are named change.

In [280]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25], "height": [165, 180]})
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180


In [281]:
df_renamed = df_students.rename(columns={"height": "height_cm", "age": "age_years"})
df_renamed

,name,age_years,height_cm
0,Ana,21,165
1,Bo,25,180


The original is unchanged until the result is assigned:

In [282]:
df_students.columns.tolist()

['name', 'age', 'height']

In [283]:
df_students = df_students.rename(columns={"height": "height_cm"})
df_students.columns.tolist()

['name', 'age', 'height_cm']

A function can rename every column at once:

In [284]:
df_students.rename(columns=str.upper)

,NAME,AGE,HEIGHT_CM
0,Ana,21,165
1,Bo,25,180


*See also:* Renaming pandas columns with a dictionary (A3) · Renaming row labels (B2)

#### Replacing all column names

**B2.04**

`df.columns = ['a', 'b', 'c']`

Assigning a list to `df.columns` allows us to rename every column at once. It is useful when the original names are missing or unusable. Alternatively, use `rename` to change only a few.

Assign a list to `df.columns` to replace **all** the names. The list must contain exactly one name per column, in order.

In [285]:
# display the example data
df = pd.DataFrame({"a": [1, 2], "b": [3, 4], "c": [5, 6]})
df

,a,b,c
0,1,3,5
1,2,4,6


In [286]:
df.columns = ["student", "years", "cm"]
df

,student,years,cm
0,1,3,5
1,2,4,6


In [287]:
df.columns = ["only", "two"]     # wrong number of names

ValueError: Length mismatch: Expected axis has 3 elements, new values have 2 elements

*See also:* Lowercasing every column name (B2)

#### Lowercasing every column name

**B2.05**

`df.columns = df.columns.str.lower()`

The `.str.lower()` method on `df.columns` allows us to standardise all column names in one line. It is useful for tidy, consistent names. Alternatively, `df.rename(columns=str.lower)` does the same.

Column names are text, so the `.str` methods work on them. Consistent lowercase names with no spaces make code simpler to type.

In [288]:
# display the example data
df = pd.DataFrame({"Student Name": ["Ana"], "AGE": [21], "Height_CM": [165]})
df

,Student Name,AGE,Height_CM
0,Ana,21,165


In [289]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df

,student_name,age,height_cm
0,Ana,21,165


*See also:* Replacing all column names (B2) · Changing case (B7)

#### Renaming row labels

**B2.06**

`df.rename(index={'a1': 'A1'})`

The `rename(index={...})` method allows us to change chosen row labels. It is useful for correcting IDs. Alternatively, assign a new list to `df.index`.

`rename(index=...)` does the same for the row labels.

In [290]:
# display the example data
df_students = pd.DataFrame({"age": [21, 25, 22]}, index=["a1", "a2", "a3"])
df_students

,age
a1,21
a2,25
a3,22


In [291]:
df_students.rename(index={"a1": "A1", "a3": "A3"})

,age
A1,21
a2,25
A3,22


*See also:* Renaming columns with a dictionary (B2) · Creating a DataFrame with row labels (B1)

#### Naming the index and column axes

**B2.07**

`df.index.name = 'id'; df.rename_axis(None, axis=1)`

Setting `df.index.name` or using `rename_axis` allows us to give (or remove) the label of the index or the columns. It is useful for tidying the stray name left by `pivot`. Alternatively, `reset_index()` turns the index into a normal column.

The index and the column labels can each have a *name*. After `pivot` the columns carry an unwanted name (like `Subject`); `rename_axis(None, axis=1)` removes it.

In [292]:
# display the example data
df = pd.DataFrame({"age": [21, 25]}, index=["a1", "a2"])
df

,age
a1,21
a2,25


In [293]:
df.index.name = "id"
df

,age
id,
a1,21
a2,25


In [294]:
df_wide = pd.DataFrame({"math": [1, 2], "art": [3, 4]}, index=[10, 11])

In [295]:
# display example dataframe
df_wide

,math,art
10,1,3
11,2,4


In [296]:
df_wide.columns.name = "Subject"
df_wide

Subject,math,art
10,1,3
11,2,4


In [297]:
df_wide.rename_axis(None, axis=1)

,math,art
10,1,3
11,2,4


*See also:* Reshaping long to wide with `set_index` and `pivot` (B6) · Setting a column as the index (B2)

### Column order

#### Reordering columns by listing them

**B2.08**

`df[['name', 'height', 'age']]`

Selecting with a list of names allows us to put columns in any order we choose. It is useful for presenting a table neatly. Alternatively, `reindex(columns=...)` does the same without raising an error for missing names.

List the columns in the required order. Leave a column out to drop it from the result.

In [298]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25], "height": [165, 180]})
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180


In [299]:
df_students[["name", "height", "age"]]

,name,height,age
0,Ana,165,21
1,Bo,180,25


*See also:* Reordering or selecting rows with `reindex` (B2) · Selecting several columns (B2) · Reordering columns with `reindex` (B2)

#### Reordering columns with `reindex`

**B2.09**

`df.reindex(columns=cols)`

The `reindex(columns=[...])` method allows us to impose an exact column order. It is useful when some columns might not exist, because it fills them with `NaN`. Alternatively, plain selection `df[[...]]` raises an error for missing names.

Like the list approach, but names that do not exist give a column of `NaN` instead of an error.

In [300]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25], "height": [165, 180]})
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180


In [301]:
df_students.reindex(columns=["height", "name", "weight"])

,height,name,weight
0,165,Ana,NaN
1,180,Bo,NaN


*See also:* Filling reindexed labels that are missing (B2)

#### Inserting or popping a column

**B2.10**

`df.insert(pos, 'col', values); df.pop('col')`

***(non-examinable)***

The `.insert()` and `.pop()` methods allow us to add a column at a chosen position or remove one and keep it. They are useful for fine control over layout. Alternatively, `df.drop(columns=...)` removes without keeping.

`insert` puts a new column at a chosen position; `pop` removes a column and returns it.

In [302]:
# display the example data
df = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25]})
df

,name,age
0,Ana,21
1,Bo,25


In [303]:
df.insert(1, "dept", ["Bio", "Chem"])

In [304]:
df.columns.tolist()

['name', 'dept', 'age']

In [305]:
col_gone = df.pop("dept")
print(col_gone.tolist(), df.columns.tolist())

['Bio', 'Chem'] ['name', 'age']


*See also:* Moving one column to the front (B2) · Reordering columns by listing them (B2) · Dropping columns (B2) · Reordering columns with `reindex` (B2)

#### Moving one column to the front

**B2.11**

`df[['c'] + [x for x in df.columns if x != 'c']]`

***(non-examinable)***

A list built from the column names allows us to move a single column to the first position. It is useful for putting an ID or key column first. Alternatively, `insert` and `pop` achieve the same.

Build a list of names with the chosen column first, then use it to select.

In [306]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25], "height": [165, 180]})
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180


In [307]:
cols = ["height"] + [c for c in df_students.columns if c != "height"]
cols

['height', 'name', 'age']

In [308]:
df_students[cols]

,height,name,age
0,165,Ana,21
1,180,Bo,25


*See also:* Reordering columns by listing them (B2)

### Moving columns to and from the index

#### Setting a column as the index

**B2.12**

`df.set_index('name')`

The `set_index()` method allows us to use a column as the row labels. It is useful for ID or date columns, so that rows can be selected with `loc`. Alternatively, `index_col=` does it while loading the file.

`set_index` turns a column into the row labels. Use it for identifier columns such as names or IDs, so that `loc` can be used with them.

In [309]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo", "Cy"], "age": [21, 25, 22]})
df_students

,name,age
0,Ana,21
1,Bo,25
2,Cy,22


In [310]:
df_by_name = df_students.set_index("name")
df_by_name

,age
name,
Ana,21
Bo,25
Cy,22


In [311]:
df_by_name.loc['Bo']

age    25
Name: Bo, dtype: int64

*See also:* Reading a CSV with a column as the index (B1) · Moving the index back into a column (B2) · Setting the index and keeping the column (B2) · Creating a DataFrame with row labels (B1)

#### Setting the index and keeping the column

**B2.13**

`df.set_index('name', drop=False)`

The `drop=False` option allows us to use a column as the index while keeping it as a column too. It is useful when the column is still needed for filtering or plotting. Alternatively, `reset_index()` brings it back later.

`drop=False` keeps the column in the table as well as using it as the index.

In [312]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo", "Cy"], "age": [21, 25, 22]})
df_students

,name,age
0,Ana,21
1,Bo,25
2,Cy,22


In [313]:
df_students.set_index("name", drop=False)

,name,age
name,,
Ana,Ana,21
Bo,Bo,25
Cy,Cy,22


*See also:* Setting a column as the index (B2) · Setting a multi-level index (B2)

#### Setting a multi-level index

**B2.14**

`df.set_index(['id', 'gender'])`

Passing a list to `set_index()` allows us to build an index from several columns. It is useful before reshaping with `melt`, so identifier columns travel with the data. Alternatively, keep them as columns and use `id_vars` in `melt`.

Several columns can form the index. This is used before `melt` so that the identifier columns are carried along (see the reshaping sheet).

In [314]:
df_bp = pd.read_csv("data/bp_experiment2.csv")

In [315]:
# display example data
df_bp.head(3)

,id,gender,bp_day1,bp_day30
0,1,f,153.0,150.0
1,2,f,143.0,139.0
2,3,m,151.0,156.0


In [316]:
df_bp.set_index(["id", "gender"]).head(3)

,,bp_day1,bp_day30
id,gender,,
1,f,153.0,150.0
2,f,143.0,139.0
3,m,151.0,156.0


*See also:* Reshaping wide to long with `set_index` and `melt` (B6) · Setting a column as the index (B2)

#### Moving the index back into a column

**B2.15**

`df.reset_index()`

The `.reset_index()` method allows us to turn the index back into an ordinary column. It is useful after `groupby`, `pivot` or `set_index`. Alternatively, `reset_index(drop=True)` discards the index instead.

The opposite of `set_index`: the index becomes an ordinary column and the rows are numbered 0, 1, 2…

In [317]:
# display the example data
df_students = pd.DataFrame({"age": [21, 25]}, index=pd.Index(["Ana", "Bo"], name="name"))
df_students

,age
name,
Ana,21
Bo,25


In [318]:
# display example data
df_students

,age
name,
Ana,21
Bo,25


In [319]:
df_students.reset_index()

,name,age
0,Ana,21
1,Bo,25


*See also:* Setting a column as the index (B2) · Setting a multi-level index (B2)

#### Discarding the old index

**B2.16**

`df.reset_index(drop=True)`

The `reset_index(drop=True)` method allows us to renumber rows 0 to n−1 and forget the old labels. It is useful after filtering or concatenating. Alternatively, `ignore_index=True` in `pd.concat` does this while combining.

After filtering or concatenating, the row labels have gaps or repeats. `drop=True` renumbers 0…n−1 and does **not** keep the old labels as a column.

In [320]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo", "Cy", "Di"], "age": [21, 25, 22, 30]})
df_students

,name,age
0,Ana,21
1,Bo,25
2,Cy,22
3,Di,30


In [321]:
df_older = df_students[df_students["age"] > 21]
df_older                     # labels 1, 2, 3

,name,age
1,Bo,25
2,Cy,22
3,Di,30


In [322]:
df_older.reset_index(drop=True)     # labels 0, 1, 2

,name,age
0,Bo,25
1,Cy,22
2,Di,30


In [323]:
df_older.reset_index()              # without drop: old labels kept as a column

,index,name,age
0,1,Bo,25
1,2,Cy,22
2,3,Di,30


*See also:* Moving the index back into a column (B2) · Filtering rows with a mask (B3) · Stacking rows (B6)

#### Keeping a result by reassigning or with `inplace`

**B2.17**

`df = df.set_index('c')  vs  inplace=True`

Reassigning (`df = df.set_index(...)`) or `inplace=True` allows us to keep the result of a method that otherwise returns a new table. It is useful whenever a method changes a table, because forgetting both means the change is lost. Alternatively, chain the methods and assign once at the end.

Most methods return a new DataFrame. Either assign the result back (preferred), or use `inplace=True`, which changes the DataFrame itself and returns `None`.

In [324]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25]})
df_students

,name,age
0,Ana,21
1,Bo,25


In [325]:
df_a = df_students.set_index("name")           # reassign
df_b = df_students.copy()
df_b.set_index("name", inplace=True)        # change in place

In [326]:
df_a.equals(df_b)

True

A common mistake:

In [327]:
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25]})

In [328]:
# display example dataframe
df_students

,name,age
0,Ana,21
1,Bo,25


In [329]:
df_students.set_index("name")          # result is displayed but thrown away

,age
name,
Ana,21
Bo,25


In [330]:
df_students.columns.tolist()    # 'name' is still a column

['name', 'age']

*See also:* Setting a column as the index (B2) · Renaming columns with a dictionary (B2) · Moving the index back into a column (B2) · Dropping columns (B2)

### Reindexing

#### Sorting by the index

**B2.18**

`df.sort_index()`

The `.sort_index()` method allows us to put rows in order of their labels. It is useful before slicing with `loc` or time-series work. Alternatively, `sort_values()` sorts by a column.

In [331]:
# display the example data
df_students = pd.DataFrame({"age": [30, 21, 25]}, index=["c", "a", "b"])
df_students

,age
c,30
a,21
b,25


In [332]:
df_students.sort_index()

,age
a,21
b,25
c,30


In [333]:
df_students.sort_index(ascending=False)

,age
c,30
b,25
a,21


*See also:* Sorting by a column (B3) · Choosing between `reset_index`, `sort_index` and `reindex` (B2) · Reordering or selecting rows with `reindex` (B2)

#### Reordering or selecting rows with `reindex`

**B2.19**

`df.reindex(['a3', 'a1', 'a2'])`

The `reindex([...])` method allows us to pick exact row labels in a chosen order. It is useful for reordering or for making sure every category appears. Alternatively, `df.loc[[...]]` selects by labels but raises an error for missing ones.

`reindex` conforms the rows to exactly the labels given, in that order. It can be used to reorder, or to select a subset.

In [334]:
# display the example data
df_students = pd.DataFrame({"age": [21, 25, 22, 30]}, index=["a1", "a2", "a3", "a4"])
df_students

,age
a1,21
a2,25
a3,22
a4,30


In [335]:
df_students.reindex(['a3', 'a1', 'a2'])

,age
a3,22
a1,21
a2,25


In [336]:
df_students.reindex(['a4', 'a1'])     # subset

,age
a4,30
a1,21


*See also:* Filling reindexed labels that are missing (B2) · Sorting by the index (B2)

#### Filling reindexed labels that are missing

**B2.20**

`df.reindex(labels, fill_value=0)`

The `fill_value` argument of `reindex` allows us to choose what appears for labels that are not in the data. It is useful for tables where a missing category should be 0, not `NaN`. Alternatively, `.fillna(0)` afterwards does the same.

New labels get `NaN` unless `fill_value` is supplied. This is useful to ensure that every category appears in a table.

In [337]:
# display the example data
s = pd.Series([5, 7], index=["a", "b"])
s

a    5
b    7
dtype: int64

In [338]:
s.reindex(['a', 'b', 'c'])

a    5.0
b    7.0
c    NaN
dtype: float64

In [339]:
s.reindex(['a', 'b', 'c'], fill_value=0)

a    5
b    7
c    0
dtype: int64

*See also:* Reordering or selecting rows with `reindex` (B2) · Filling missing values (B5) · Checking the column types (B1)

#### Choosing between `reset_index`, `sort_index` and `reindex`

**B2.21**

`reset_index(drop=True) vs reindex vs sort_index`

Comparing `reset_index(drop=True)`, `sort_index()` and `reindex()` allows us to choose the right tool for each tidying job, because they look similar but do different things.

- **`reset_index(drop=True)`** – forget the labels, renumber 0…n−1.
- **`sort_index()`** – keep the labels, put them in order.
- **`reindex([...])`** – choose exactly which labels and in what order (adds missing ones as `NaN`).

In [340]:
# display the example data
df_students = pd.DataFrame({"age": [21, 25, 22, 30]}, index=["a1", "a2", "a3", "a4"])
df_students

,age
a1,21
a2,25
a3,22
a4,30


In [341]:
df = df_students.sort_values("age", ascending=False)
print(f"after sorting by age     : {df.index.tolist()}")

after sorting by age     : ['a4', 'a2', 'a3', 'a1']


In [342]:
print(f"sort_index()             : {df.sort_index().index.tolist()}")

sort_index()             : ['a1', 'a2', 'a3', 'a4']


In [343]:
print(f"reset_index(drop=True)   : {df.reset_index(drop=True).index.tolist()}")

reset_index(drop=True)   : [0, 1, 2, 3]


In [344]:
print(f"reindex(['a2', 'a1'])    : {df.reindex(['a2', 'a1']).index.tolist()}")

reindex(['a2', 'a1'])    : ['a2', 'a1']


*See also:* Discarding the old index (B2) · Sorting by the index (B2) · Reordering or selecting rows with `reindex` (B2)

### Dropping columns and rows

#### Dropping columns

**B2.22**

`df.drop(columns=['age'])`

The `drop(columns=...)` method allows us to remove columns we do not need. It is useful for tidying a wide table. Alternatively, select only the columns to keep with `df[[...]]`.

`drop` returns a new DataFrame without the named columns.

In [345]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25], "height": [165, 180]})
df_students

,name,age,height
0,Ana,21,165
1,Bo,25,180


In [346]:
df_students.drop(columns=["age", "height"])

,name
0,Ana
1,Bo


*See also:* Dropping rows by label (B2) · Choosing rows or columns with `axis` (B2)

#### Dropping rows by label

**B2.23**

`df.drop(index=['a1'])`

The `drop(index=...)` method allows us to remove specific rows by label. It is useful for deleting known bad records. Alternatively, filter with a mask to keep only the good rows.

In [347]:
# display the example data
df_students = pd.DataFrame({"age": [21, 25, 22]}, index=["a1", "a2", "a3"])
df_students

,age
a1,21
a2,25
a3,22


In [348]:
df_students.drop(index=["a1", "a2"])

,age
a3,22


*See also:* Dropping columns (B2) · Removing rows with missing values (B5) · Choosing rows or columns with `axis` (B2) · Filtering rows with a mask (B3)

#### Choosing rows or columns with `axis`

**B2.24**

`df.drop('age', axis=1)`

The `axis` argument allows us to say whether `drop` should work on rows (0) or columns (1). It is useful when reading other people's code, which often sets `axis` without comment. Alternatively, `columns=` and `index=` are clearer.

`axis=1` means columns and `axis=0` means rows. The `columns=` / `index=` keywords are clearer, but `axis` also appears in other code.

In [349]:
# display the example data
df_students = pd.DataFrame({"name": ["Ana", "Bo"], "age": [21, 25]})
df_students

,name,age
0,Ana,21
1,Bo,25


In [350]:
df_students.drop('age', axis=1).columns.tolist()    # drop a column

['name']

In [351]:
df_students.drop(0, axis=0).index.tolist()          # drop row 0

[1]

*See also:* Dropping columns (B2) · Dropping rows by label (B2)

### Changing types

#### Changing a column's type

**B2.25**

`df['age'].astype(float)`

The `.astype()` method allows us to convert a column to another type, such as int or float. It is useful when numbers are stored as text. Alternatively, `pd.to_numeric()` is more forgiving of bad values.

Before: whole numbers (`int64`).

In [352]:
# display the example data
df_students = pd.DataFrame({"age": [21, 25, 22]})
df_students

,age
0,21
1,25
2,22


In [353]:
df_students['age'].dtype

dtype('int64')

In [354]:
df_students["age"].tolist()

[21, 25, 22]

After `astype(float)`: the same numbers, now with decimals (`float64`).

In [355]:
df_students["age"] = df_students["age"].astype(float)
df_students['age'].dtype

dtype('float64')

In [356]:
df_students["age"].tolist()

[21.0, 25.0, 22.0]

Text that looks like numbers must be converted before calculations can be made with it:

In [357]:
s = pd.Series(["10", "20", "30"])

In [358]:
# display example series
s

0    10
1    20
2    30
dtype: object

In [359]:
print(f"before: {s.dtype} {s.tolist()}")      # text: quotes

before: object ['10', '20', '30']


In [360]:
col_s_int = s.astype(int)
print(f"after : {col_s_int.dtype} {col_s_int.tolist()}")

after : int64 [10, 20, 30]


In [361]:
col_s_int.sum()

np.int64(60)

*See also:* Checking the column types (B1) · Converting between types (A1) · Converting a column to string type (B7) · Checking a DataFrame after each step (B1)